In [72]:
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")

# row_limit = 500

df = pd.read_csv("datathon_year_3.csv")

# rows, cols
print(df.shape)
print(df.dtypes)

(3995, 15)
Idea #                                int64
Original Title                          str
Challenge                               str
Solution                                str
Audience                                str
Borough                                 str
Neighborhood(s)                         str
Host of Idea Generation Workshop        str
Impact Area                             str
Sub-Category                            str
Type of Idea                            str
Idea Zip Code                       float64
Idea Generation Session Zip Code    float64
Idea Source                             str
Insights                                str
dtype: object


In [73]:
# !pip
# %pip

# preview of first 5 rows
print(df.head(5))

   Idea #                          Original Title  \
0      14  Pathways to Success: Bilingual Support   
1     111                    Youth Launch Project   
2     209            Housing workshop for seniors   
3     289             Teen Employee Connect (TEC)   
4     348                        "Your not alone"   

                                           Challenge  \
0  Teens of immigrants struggle with the college ...   
1  A challenge in Harlem is helping students lear...   
2  For some seniors with limited credit and no in...   
3  Many teens do not have methods of income in Ca...   
4  Many younger parents and parents with no help ...   

                                            Solution  \
0  A mentorship program that would offer bilingua...   
1  A youth enterprenuership program that teaches ...   
2  There is an urgent need for senior apartments,...   
3  develop a program to connect teens with availa...   
4  More parenting coaching classes for young adul...   

       

In [74]:
# check for NA values
print(df.isna().sum())

Idea #                                 0
Original Title                        41
Challenge                              6
Solution                              13
Audience                               1
Borough                                0
Neighborhood(s)                      201
Host of Idea Generation Workshop     309
Impact Area                            4
Sub-Category                         474
Type of Idea                         703
Idea Zip Code                       3709
Idea Generation Session Zip Code     313
Idea Source                            0
Insights                             838
dtype: int64


In [75]:
# Check exact duplicate based on cols subset
duplicated_rows = df[df.duplicated(subset=["Solution"])]
print(duplicated_rows)

      Idea #                                     Original Title  \
743     1777                                                NaN   
787     1931                               Unity Beyond Borders   
1146    4495  "We Grow Life" is a citywide initiative to cul...   
1208    4340  "We Grow Life" is a citywide initiative to cul...   
1404    1067                          Disability Center For All   
1485    1529         incarcerated lack skills and opportunities   
1488    1539                            The Clean Streets act\n   
1926     129                   Soup Kitchen and Laundry Service   
2030     296                                       Gun Violence   
2048     335                                         chip away    
2071     381                              viviendas enjecientes   
2072     382                     Vivienda para los envejesiente   
2073     383                             Vivienda Envejecientes   
2074     384                             Vivienda Envejeciente

In [76]:
# remove characters from rows that start with bad chars
df['Challenge'] = df['Challenge'].str.lstrip("\"'@-=")
# below removes whitespace but technically whitespace doesn't matter
# df['Challenge'] = df['Challenge'].str.strip()

In [ ]:
# stop_words = spacy.lang.en.stop_words.STOP_WORDS
# print('Number of stopwords: %d' % len(stop_words))
# print(list(stop_words))

Number of stopwords: 326
['within', 'meanwhile', 'you', 'itself', 'to', 'ourselves', 'used', 'not', 'my', 'anyway', 'up', 'each', 'if', 'during', 'others', 'here', 'nevertheless', 'eleven', 'now', 'seems', 'full', 'also', 'doing', 'ever', 'on', 'was', 'regarding', 'whereafter', 'any', 'besides', 'formerly', 'no', 'front', 'seemed', 'hence', "'re", 'this', 'just', 'had', 'under', 'a', 'whole', 'enough', 'he', 'beside', 'latter', 'however', 'next', 'be', 'please', 'beyond', 'herein', 'are', 'been', 'five', 'has', 'name', 'yourselves', 'among', 'call', 'whose', 'being', 'your', 'rather', 'take', 'nowhere', 'once', 'these', 'therefore', 'many', 'thereupon', 'much', 'what', 'her', 'could', 'toward', '’ll', 'do', 'moreover', 'our', 'cannot', 'since', 'themselves', 'upon', 'an', 'or', 'none', 'whom', 'hers', '’ve', 'too', 'whereby', 'but', 'one', 'across', 'never', 'first', "'ll", 'former', "'s", 'n‘t', 'against', 'about', 'so', '‘re', 'the', 'whatever', 'every', 'were', 'even', 'of', 'whethe

In [79]:
# noun chunk extraction
def extract_noun_chunk(text):
    doc = nlp(text.lower())
    return " | ".join([chunk.text for chunk in doc.noun_chunks])

df['Challenge_noun_chunk'] = df['Challenge'].apply(extract_noun_chunk)
df['Solution_noun_chunk'] = df['Solution'].apply(extract_noun_chunk)
temp_df_nc = df[['Challenge_noun_chunk', 'Solution_noun_chunk']]
print(temp_df_nc.head(10))

AttributeError: 'float' object has no attribute 'lower'

In [ ]:
# lemmatize function
def clean_lemmatize(text):
    doc = nlp(text.lower())
    return " ".join([
        token.lemma_
        for token in doc
        # punctuation   
        if not token.is_punct   
        # spaces 
        and not token.is_space   
        # stopwords
        #and not token.is_stop  
    ])

df['Challenge_lem'] = df['Challenge'].apply(clean_lemmatize)
df['Solution_lem'] = df['Solution'].apply(clean_lemmatize)
temp_df = df[['Challenge_lem', 'Solution_lem']]
print(temp_df.head(20))

                                        Challenge_lem  \
0   teen immigrant struggle college application pr...   
1   challenge harlem help student learn start busi...   
2   senior limited credit income find housing majo...   
3                         teen method income canarsie   
4          young parent parent help grow parent selve   
5   school rigid structure leave time extra curric...   
6   club elementary school grade like high school ...   
7   pakistani woman immigrant unemployed support f...   
8   lack relevant work opportunity teen youth word...   
9                              people drown know swim   
10  lack access free financial literacy education ...   
11  new york city housing crisis landlord advantag...   
12                youth outlet express gift misdirect   
13                              people cook mean feed   
14  folk limit access housing struggle food insecu...   
15  youth employment trade programming outreach of...   
16  awareness autism student di

In [ ]:
# dependency parsing

# def show_dependencies(text):
#     doc = nlp(text)
#     for token in doc:
#         print(token.text, token.dep_, token.head.text)

# subject object verb extraction
def extract_svo(text):
    doc = nlp(text)
    svos = []
    
    for token in doc:
        if token.pos_ == "VERB":
            subjects = [child for child in token.children if child.dep_ in ["nsubj", "nsubjpass"]]
            objects = [child for child in token.children if child.dep_ in ["dobj", "attr", "oprd"]]
            
            # include prepositional objects
            for prep in [child for child in token.children if child.dep_ == "prep"]:
                objects.extend([child for child in prep.children if child.dep_ == "pobj"])
            
            for subj in subjects:
                for obj in objects:
                    svos.append((subj.text, token.lemma_, obj.text))
    
    return svos

df["svos_problems"] = df["Challenge"].apply(extract_svo)
df["svos_solutions"] = df["Solution"].apply(extract_svo)

# test for checking visualization of SVO on row 5 (cant use on whole column)
# for text in df["Community problems"].head(5):
#     doc = nlp(text)
#     displacy.render(doc, style="dep", jupyter=True)

KeyError: 'Community problems'